# LLM Inference Mechanics

> **Source notes:** `ch02-llm-inference-mechanics`

This notebook explores the performance characteristics of LLM inference:
- **KV Cache Speedup** — measure the impact of key-value caching on generation speed
- **Prefill vs Decode Latency** — understand first-token vs per-token costs
- **Batching Throughput** — observe how batch processing affects tokens/second

All experiments use **LLaMA 3.2 3B** via HuggingFace Transformers (local, no API key needed). First-time download: ~6GB.

## 0 · Environment Setup

Install required packages and import libraries.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "transformers", "torch", "matplotlib", "numpy", "-q"], check=True)
print("✓ Packages installed")

In [ ]:
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

print("✓ Libraries imported")
print(f"  PyTorch version: {torch.__version__}")
print(f"  Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 1 · KV Cache Speedup Benchmark

**Key-value (KV) caching** is a critical optimization in autoregressive generation. Without it, each new token requires recomputing attention for the entire sequence. With caching, we only compute attention for the new token and reuse previous keys/values.

**Your task:**
1. Load LLaMA 3.2 3B small
2. Generate 100 tokens WITHOUT KV caching (set `use_cache=False`)
3. Generate 100 tokens WITH KV caching (set `use_cache=True`)
4. Measure time per token for both approaches
5. Calculate the speedup ratio

**Expected behavior:** KV caching should provide 2-5x speedup for moderate sequence lengths.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `tokenizer` using `from_pretrained()`
# 2. Compute `prompt` using `encode()`
# 3. Compute `start_time` using `time()`
# 4. Process data
# 5. Compute `start_time` using `time()`
# 6. Process data
# 7. Compute `speedup`
#
# Hint:
#    tokenizer = AutoTokenizer.from_pretrained(???)
#    model = AutoModelForCausalLM.from_pretrained(???)
#    input_ids = tokenizer.encode(???)
#    start_time = time.time(???)

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
methods = ['Without KV Cache', 'With KV Cache']
times = [time_per_token_no_cache * 1000, time_per_token_cache * 1000]  # Convert to ms
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(methods, times, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('Time per Token (ms)', fontsize=12, fontweight='bold')
ax.set_title('KV Cache Speedup Benchmark', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}ms',
            ha='center', va='bottom', fontweight='bold')

# Add speedup annotation
ax.text(0.5, max(times) * 0.9, f'Speedup: {speedup:.2f}x',
        ha='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.show()

## 2 · Prefill vs Decode Latency

LLM inference has two distinct phases:
- **Prefill (encode)**: Process the entire prompt in one forward pass. This is the "first token latency".
- **Decode**: Generate tokens one at a time autoregressively.

Prefill is expensive (grows with prompt length), but decode latency per token is more stable.

**Your task:**
1. Generate text with varying prompt lengths: 100, 500, 1000, 2000 tokens
2. For each prompt length, measure:
 - Prefill time (time to generate the first token)
 - Average decode time per token (for next 50 tokens)
3. Plot both metrics vs prompt length

**Expected behavior:** Prefill time grows with prompt length, decode time stays relatively constant.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `prompt_lengths`
# 2. Compute `base_string`
# 3. Call `encode()` to produce the result
# 4. Process data
# 5. Call `time()` to produce the result
# 6. Process data
#
# Hint:
#    input_ids = tokenizer.encode(???)
#    start_time = time.time(???)
#    _ = model.generate(???)
#    total_decode_time = time.time(???)

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(prompt_lengths, prefill_times, marker='o', linewidth=2,
        markersize=8, label='Prefill Time (First Token)', color='#e74c3c')
ax.plot(prompt_lengths, decode_times, marker='s', linewidth=2,
        markersize=8, label='Decode Time (Per Token)', color='#3498db')

ax.set_xlabel('Prompt Length (tokens)', fontsize=12, fontweight='bold')
ax.set_ylabel('Latency (ms)', fontsize=12, fontweight='bold')
ax.set_title('Prefill vs Decode Latency', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

# Add annotations for the longest prompt
ax.annotate(f'{prefill_times[-1]:.1f}ms',
            xy=(prompt_lengths[-1], prefill_times[-1]),
            xytext=(10, 10), textcoords='offset points',
            fontweight='bold', color='#e74c3c')
ax.annotate(f'{decode_times[-1]:.1f}ms',
            xy=(prompt_lengths[-1], decode_times[-1]),
            xytext=(10, -15), textcoords='offset points',
            fontweight='bold', color='#3498db')

plt.tight_layout()
plt.show()

## 3 · Batching Throughput

**Batching** allows the model to process multiple requests in parallel, amortizing compute overhead. Throughput (tokens/second) typically increases with batch size, though individual request latency may increase.

**Your task:**
1. Generate text with batch_size = 1, 4, 8, 16
2. For each batch size:
 - Create a batch of prompts (repeat the same prompt)
 - Generate 50 tokens per prompt
 - Measure total time
 - Calculate throughput = (batch_size × 50) / total_time
3. Plot throughput vs batch size

**Expected behavior:** Throughput increases with batch size (up to hardware limits), showing better GPU utilization.

**Note:** This exercise is most meaningful on GPU. On CPU, benefits may be limited.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `batch_sizes`
# 2. Compute `base_prompt`
# 3. Process data
# 4. Call `tokenizer()` to produce the result
# 5. Call `time()` to produce the result
# 6. Call `append()` to produce the result
# 7. Process data
#
# Hint:
#    start_time = time.time(???)
#    outputs = model.generate(???)
#    total_time = time.time(???)

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(range(len(batch_sizes)), throughputs,
              color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'],
              alpha=0.8, edgecolor='black')

ax.set_xlabel('Batch Size', fontsize=12, fontweight='bold')
ax.set_ylabel('Throughput (tokens/sec)', fontsize=12, fontweight='bold')
ax.set_title('Batching Throughput Analysis', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(batch_sizes)))
ax.set_xticklabels(batch_sizes)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}',
            ha='center', va='bottom', fontweight='bold')

# Add efficiency gain annotation
if len(throughputs) > 1:
    gain = throughputs[-1] / throughputs[0]
    ax.text(0.5, max(throughputs) * 0.85,
            f'Batch-{batch_sizes[-1]} is {gain:.2f}x faster than Batch-{batch_sizes[0]}',
            ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5),
            transform=ax.transAxes)

plt.tight_layout()
plt.show()

**Observations to note:**
- Temperature 0.0 produces the same output every time (deterministic)
- Higher temperatures make output more creative but potentially less coherent
- Top-p (nucleus sampling) is more adaptive than top-k
- Lower top-p/top-k values make generation more conservative and focused

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Compute `top_k_values` using `values()`
# 3. Process data
# 4. Call `manual_seed()` to produce the result
# 5. Call `no_grad()` to produce the result
# 6. Call `decode()` to produce the result
# 7. Call `focused()` to produce the result
#
# Hint:
#    output = model.generate(???)
#    generated_text = tokenizer.decode(???)
#    tokenizer.decode(???)

## 4 · Sampling Parameters — Temperature, Top-p, Top-k

**Sampling parameters** control how the model selects the next token from the probability distribution. Understanding their effects is crucial for controlling generation quality and creativity.

**Your task:**
1. Generate the same prompt with different temperature values: 0.0, 0.5, 1.0, 1.5
2. Generate with different top-p values: 0.5, 0.9, 0.95, 1.0 (nucleus sampling)
3. Generate with different top-k values: 10, 50, 100 (keep only top-k most likely tokens)
4. Compare outputs to understand their effects

**Expected behavior:**
- Temperature 0.0 = deterministic (same output every time)
- Higher temperature = more creative/random
- Lower top-p = more focused on likely tokens
- Lower top-k = more conservative generation

## Summary

Key takeaways from this notebook:

| Optimization | Impact | When to use |
|---|---|---|
| **KV Cache** | 2-5x speedup | Always enabled (default in production) |
| **Prefill optimization** | Reduces first-token latency | Critical for interactive apps, long prompts |
| **Batching** | Increases throughput | High-load inference servers, batch processing |
| **Temperature** | Controls randomness/creativity | Tune per use case (0.0 = factual, 1.0+ = creative) |
| **Top-p** | Adaptive token filtering | Default choice (0.9) for quality |
| **Top-k** | Fixed-size token filtering | Less common, less adaptive |

**Production implications:**
- KV cache is essential — without it, generation becomes quadratically expensive
- Prefill dominates latency for long prompts (>1000 tokens) — consider prompt compression or retrieval
- Batching maximizes GPU utilization but increases per-request latency — tune based on SLA requirements
- Sampling parameters control output quality: low temperature for facts, high for creativity

**Next:** Explore quantization and model compression techniques for further efficiency gains.